# EcoData · S11 — BI/ML distribuido con Spark (dimensión clima)
**Alahin Reyme Ticona Veliz** · Proyecto Sello EcoData (dimensión **clima / temperatura**, Juliaca) · Big Data, UPeU

Dos partes, igual que la guía:

| Parte | Guía S11 | EcoData · clima |
|---|---|---|
| **A · BI batch** | KPIs del Parquet de S9, un solo `PUT` al Pushgateway | KPIs del Parquet de **S9 clima** (`s09_clima_observabilidad`): volumen, ratio de válidos, latencia **y KPIs de negocio** (temperatura mínima, horas bajo 0 °C) → job `lambda26_bi_flujo_clima` |
| **B · ML en vivo** | modelo de S10 redesplegado, un `PUT` por micro-lote | modelo de **S10 clima** (`modelo_temp_s10`) sobre `ecodata-clima-s10` → job `lambda26_ml_streaming_clima` (temperatura real, predicha y bandera de helada prevista) |
| Tablero | Stat (batch) + Time series (en vivo) | `s11_dashboard_clima.json` |

**Requisitos:** haber ejecutado **S9** (genera el Parquet) y **S10** (guarda el modelo) en esta misma carpeta, con Kafka, Prometheus y Pushgateway arriba. Cierra otros cuadernos de Spark.

> **Honestidad sobre Pushgateway:** la Parte A es el caso correcto (job que termina). La Parte B es un **atajo** (job continuo): si el productor se detiene, el último valor queda congelado en el Pushgateway sin ninguna señal. En EcoData ese riesgo se mitiga con el *semáforo de frescura* (`time() - push_time_seconds`) y alertas con `absent()`; la alternativa recomendada sería un endpoint `/metrics` propio.

## 0. Preparación

In [1]:
import sys, json, time, threading, urllib.request, urllib.parse, os
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import clima_comun as clima
import openmeteo_clima_a_kafka as pub_clima
from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS_CANDIDATOS = ["http://prometheus:9090", "http://lambda26-prometheus:9090"]
PKG = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
HISTORICO = "/tmp/s12/clima_juliaca_2023.json"      # temperatura, humedad y viento horarios reales (Open-Meteo), ya en el proyecto

def crear_topic(nombre, particiones=3):
    adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
    try: adm.create_topics([NewTopic(nombre, num_partitions=particiones, replication_factor=1)]); print("topic creado:", nombre)
    except TopicAlreadyExistsError: print("el topic ya existía:", nombre)
    adm.close()

def prom(q):
    for base in PROMETHEUS_CANDIDATOS:
        try:
            u = f"{base}/api/v1/query?" + urllib.parse.urlencode({"query": q})
            return json.load(urllib.request.urlopen(u, timeout=4))["data"]["result"]
        except Exception:
            continue
    return None

def publicar_paced(topic, eventos, pausa_s=2.0):
    """Publica un evento cada pausa_s segundos en segundo plano (la celda no se bloquea)."""
    def run():
        p = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all", key_serializer=str.encode,
                          value_serializer=lambda v: v if isinstance(v, bytes) else json.dumps(v).encode())
        for e in eventos:
            p.send(topic, key=e["sensorId"], value=e); p.flush(); time.sleep(pausa_s)
        p.close()
    th = threading.Thread(target=run, daemon=True); th.start(); return th

def desde_archivo(path=HISTORICO):
    """[(ts_ms, temp, hum, viento)] ordenado y sin huecos, del histórico real del proyecto."""
    return clima.leer_json(path)

def serie_clima(min_horas=96):
    """Primero la API (horas recientes); si trae pocas (suele dar ~50 h), el histórico real. Devuelve (serie, origen)."""
    try:
        s = pub_clima.desde_api(); origen = "openmeteo-api"
    except Exception as ex:
        print("API no disponible:", type(ex).__name__); s, origen = [], "openmeteo-api"
    if len(s) < min_horas:
        print(f"La API trajo {len(s)} horas; uso el histórico real de Juliaca (origen openmeteo-replay)")
        s, origen = desde_archivo(), "openmeteo-replay"
    return s, origen

TOPIC = "ecodata-clima-s10"
RUTA_S9 = "./artifacts/s09_clima_observabilidad"
RUTA_MODELO = "./artifacts/models/modelo_temp_s10"
print("Parquet S9:", os.path.exists(RUTA_S9), "· Modelo S10:", os.path.exists(RUTA_MODELO))

Parquet S9: True · Modelo S10: True


In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
esquema_evento = (StructType().add("tipoEvento", StringType()).add("sensorId", StringType())
                  .add("temp", DoubleType()).add("hum", DoubleType()).add("viento", DoubleType())
                  .add("origen", StringType()).add("timestamp", LongType()))
spark = (SparkSession.builder.appName("ecodata-clima-s11").master("local[2]")
         .config("spark.jars.packages", PKG)
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.port", "4062")          # puertos 4060-4063: no chocan con 4040 (S12) ni con 4050-4053 (aire)
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)
from pyspark.ml import PipelineModel

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f3da0826-76e5-4c96-80bd-8feb2b6e923f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in spark-list
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.

Spark 4.2.0


## 1. Parte A — KPIs del flujo de eventos (3.2–3.3)

In [3]:
eventos_s9 = spark.read.parquet(RUTA_S9)
kpi = eventos_s9.agg(F.count("*").alias("total"),
                     F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
                     F.avg("latencyMs").alias("avgLatencyMs"),
                     F.percentile_approx("latencyMs", 0.95).alias("p95LatencyMs"),
                     F.min("temp").alias("tempMin"),
                     F.sum(F.when(F.col("temp") < 0, 1).otherwise(0)).alias("horasHelada")).first()
print(f"Eventos procesados:  {kpi['total']}")
print(f"Ratio de válidos:    {kpi['ratioValidos']:.4f}")
print(f"Latencia promedio:   {kpi['avgLatencyMs']:.1f} ms  (p95 = {kpi['p95LatencyMs']} ms)")
print(f"KPI de negocio:      temperatura mínima {kpi['tempMin']:.1f} °C · horas bajo 0 °C: {kpi['horasHelada']}")

[Stage 1:>                                                          (0 + 2) / 2]

Eventos procesados:  45
Ratio de válidos:    1.0000
Latencia promedio:   2561.6 ms  (p95 = 4953 ms)
KPI de negocio:      temperatura mínima 2.0 °C · horas bajo 0 °C: 0


## 2. Publicar los KPIs una sola vez en el Pushgateway y verificarlos (3.4)

In [4]:
metricas = {"lambda26_bi_clima_eventos_total": kpi["total"],
            "lambda26_bi_clima_validos_ratio": kpi["ratioValidos"],
            "lambda26_bi_clima_latencia_promedio_ms": kpi["avgLatencyMs"],
            "lambda26_bi_clima_latencia_p95_ms": kpi["p95LatencyMs"],
            "lambda26_bi_clima_temp_min_c": kpi["tempMin"],
            "lambda26_bi_clima_horas_helada": kpi["horasHelada"]}
cuerpo = "".join(f"# TYPE {n} gauge\n{n} {v}\n" for n, v in metricas.items())
req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/job/lambda26_bi_flujo_clima/instance/s11_batch", data=cuerpo.encode(), method="PUT")
print("Pushgateway respondió:", urllib.request.urlopen(req).status)

Pushgateway respondió: 200


In [5]:
time.sleep(20)       # al menos un scrape_interval (15 s)
for n in metricas:
    r = prom(n); print(f"{n}: {r[0]['value'][1] if r else 'SIN DATOS (espera otro scrape y repite)'}")

lambda26_bi_clima_eventos_total: 45
lambda26_bi_clima_validos_ratio: 1
lambda26_bi_clima_latencia_promedio_ms: 2561.6
lambda26_bi_clima_latencia_p95_ms: 4953
lambda26_bi_clima_temp_min_c: 2
lambda26_bi_clima_horas_helada: 0


## 3. Parte B — redesplegar el modelo de S10 en vivo, sin reentrenar (3.6–3.8)

In [6]:
crear_topic(TOPIC)
modelo = PipelineModel.load(RUTA_MODELO)
print("Modelo cargado desde:", RUTA_MODELO)

el topic ya existía: ecodata-clima-s10
Modelo cargado desde: ./artifacts/models/modelo_temp_s10


In [7]:
crudo = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
         .option("subscribe", "ecodata-clima-s10").option("startingOffsets", "latest").option("failOnDataLoss", "false").load())
eventos = (crudo.select("topic", "partition", "offset", F.col("timestamp").alias("kafkaTimestamp"),
                        F.from_json(F.col("value").cast("string"), esquema_evento).alias("e"))
           .select("topic", "partition", "offset", "kafkaTimestamp", "e.*"))
pred_stream = modelo.transform(eventos).withColumn("pred_temp", F.col("temp") + F.col("prediction"))
RUTA_PRED = "./artifacts/s11_predicciones_stream_clima"; CHK = "./artifacts/chk_s11_clima"
JOB_ML = f"{PUSHGATEWAY}/metrics/job/lambda26_ml_streaming_clima/instance"
publicaciones = []
def push_en_vivo(sensor, real, pred):
    cuerpo = ("# TYPE lambda26_ml_temp_real gauge\n" f"lambda26_ml_temp_real {real}\n"
              "# TYPE lambda26_ml_temp_prediccion_siguiente gauge\n" f"lambda26_ml_temp_prediccion_siguiente {pred}\n"
              "# TYPE lambda26_ml_helada_prevista gauge\n" f"lambda26_ml_helada_prevista {1 if pred < 0 else 0}\n")
    urllib.request.urlopen(urllib.request.Request(f"{JOB_ML}/{sensor}", data=cuerpo.encode(), method="PUT")).close()

def por_microlote(dfm, bid):
    n = dfm.count()
    if n == 0:
        publicaciones.append(dict(batchId=bid, filas=0)); return
    dfm.select(F.lit(bid).alias("batchId"), "sensorId", "timestamp", "temp", F.col("pred_temp").alias("pred")).write.mode("append").parquet(RUTA_PRED)
    for r in dfm.select("sensorId", "temp", "pred_temp").collect():
        push_en_vivo(r["sensorId"], r["temp"], r["pred_temp"])
    publicaciones.append(dict(batchId=bid, filas=n, status="guardado y publicado en vivo"))

if "q" in dir() and q.isActive: q.stop()
q = pred_stream.writeStream.foreachBatch(por_microlote).option("checkpointLocation", CHK).start()
print("consulta activa:", q.isActive)

consulta activa: True


In [8]:
serie = desde_archivo()
i_min = min(range(len(serie)), key=lambda i: serie[i][1])           # tramo con la hora más fría: así se ve cruzar el umbral de helada
ini = max(24, min(i_min - 30, len(serie) - 60)); tail = serie[ini:ini + 60]
h = publicar_paced(TOPIC, [clima.evento(t, x, hh, v, "openmeteo-replay") for t, x, hh, v in tail], pausa_s=3.0)
print(len(tail), "eventos en camino (1 cada 3 s ≈ 3 min). No detengas la consulta.")

60 eventos en camino (1 cada 3 s ≈ 3 min). No detengas la consulta.


## 4. Confirmar que es en vivo: la misma consulta PromQL, dos veces (3.9)

In [9]:
q1 = prom("lambda26_ml_temp_real"); print("Primera lectura:", q1[0]["value"][1] if q1 else None, "·", time.strftime("%H:%M:%S"))
time.sleep(20)
q2 = prom("lambda26_ml_temp_real"); print("Segunda lectura:", q2[0]["value"][1] if q2 else None, "·", time.strftime("%H:%M:%S"))
print("¡cambió!" if q1 and q2 and q1[0]["value"][1] != q2[0]["value"][1] else "Igual: espera otros 20 s o revisa que el productor siga publicando")
print(publicaciones[-3:])

Primera lectura: 0.9 · 04:42:55


Segunda lectura: -0.3 · 04:43:15
¡cambió!
[{'batchId': 9, 'filas': 1, 'status': 'guardado y publicado en vivo'}, {'batchId': 10, 'filas': 1, 'status': 'guardado y publicado en vivo'}, {'batchId': 11, 'filas': 1, 'status': 'guardado y publicado en vivo'}]


**Tablero (3.11):** importa `s11_dashboard_clima.json` en Grafana (Dashboards → New → Import). Pon *Last 5 minutes* y refresh *5s* mientras corre la publicación y captura el panel en vivo.

## 5. Cerrar, auditar y limpiar el valor congelado (3.8, 3.10)

In [10]:
h.join(timeout=240); time.sleep(10)
q.stop()
# Hallazgo (2.3 de la guía): con el productor detenido, el Pushgateway sigue mostrando el último valor
c1 = prom("lambda26_ml_temp_real"); time.sleep(20); c2 = prom("lambda26_ml_temp_real")
print("Con el productor detenido →", c1[0]["value"][1] if c1 else None, "|", c2[0]["value"][1] if c2 else None, "(idéntico = congelado, no en vivo)")

Con el productor detenido → 11.4 | 11.4 (idéntico = congelado, no en vivo)


In [11]:
res = spark.read.parquet(RUTA_PRED).orderBy("timestamp").toPandas()
res["real_siguiente"] = res["temp"].shift(-1); aud = res.dropna()
print(f"Predicciones comparables: {len(aud)} · MAE en producción (stream): {(aud['pred'] - aud['real_siguiente']).abs().mean():.4f} °C"
      f" · persistencia: {(aud['temp'] - aud['real_siguiente']).abs().mean():.4f} °C")
res.head(8)

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
[Stage 319:>                                                        (0 + 2) / 2]

Predicciones comparables: 59 · MAE en producción (stream): 0.5434 °C · persistencia: 1.7763 °C


,batchId,sensorId,timestamp,temp,pred,real_siguiente
0,1,openmeteo-clima-juliaca,1686286800000,2.0,1.280726,1.4
1,2,openmeteo-clima-juliaca,1686290400000,1.4,0.944553,0.9
2,3,openmeteo-clima-juliaca,1686294000000,0.9,0.503572,2.0
3,4,openmeteo-clima-juliaca,1686297600000,2.0,0.687555,1.1
4,5,openmeteo-clima-juliaca,1686301200000,1.1,-0.367402,-1.8
5,6,openmeteo-clima-juliaca,1686304800000,-1.8,-2.063352,-2.3
6,7,openmeteo-clima-juliaca,1686308400000,-2.3,-0.103456,-0.3
7,8,openmeteo-clima-juliaca,1686312000000,-0.3,3.282639,3.5


In [12]:
# Limpieza: borrar el grupo congelado para que Grafana no siga mostrando un valor viejo
sensor = res["sensorId"].iloc[0]
urllib.request.urlopen(urllib.request.Request(f"{JOB_ML}/{sensor}", method="DELETE")).close()
print("Grupo borrado del Pushgateway:", sensor)

Grupo borrado del Pushgateway: openmeteo-clima-juliaca


### Qué capturar (con **reloj del sistema y tu perfil visibles**)
1. Sección 1 (KPIs) y sección 2 (PUT 200 + verificación en Prometheus).
2. Sección 3: consulta activa y `publicaciones` con micro-lotes publicados.
3. Sección 4: las dos consultas PromQL con valores distintos y hora distinta.
4. Grafana: tablero con panel batch y panel en vivo.
5. Sección 5: valor congelado (hallazgo) y MAE en producción.

In [13]:
spark.stop()